# 09 — Comparativo entre grupos

**Librerias usadas:** pandas, numpy, scipy

In [ ]:
import sys, os
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'solineras'))
os.chdir(ROOT)
DATA = ROOT / 'data'

f = DATA / 'clean' / 'encuestas_consolidadas.parquet'
if not f.exists():
    from consolidar import consolidar
    consolidar()
df = pd.read_parquet(f)
print('Dataset:', df.shape)
if 'tipo_entidad' in df.columns:
    print(df['tipo_entidad'].value_counts())

## 1. Bootstrap para medianas

In [ ]:
def bootstrap_median_ci(serie, n_iter=1000, alpha=0.05):
    serie = np.asarray(serie, dtype=float)
    serie = serie[~np.isnan(serie)]
    if len(serie) < 3:
        return None
    meds = [np.median(np.random.choice(serie, size=len(serie), replace=True))
            for _ in range(n_iter)]
    return {'mediana': float(np.median(serie)),
            'ic_bajo': float(np.percentile(meds, 100*alpha/2)),
            'ic_alto': float(np.percentile(meds, 100*(1-alpha/2))),
            'n': len(serie)}

VARS = ['potencia_pico_kwp', 'consumo_total_anual_kwh', 'pr_promedio',
        'satisfaccion', 'kwh_por_kwp', 'antiguedad_anios', 'ratio_bateria_panel']

if 'tipo_entidad' in df.columns:
    filas = []
    for tipo, g in df.groupby('tipo_entidad'):
        for var in VARS:
            if var in g.columns:
                ci = bootstrap_median_ci(g[var].dropna())
                if ci:
                    filas.append({'tipo': tipo, 'variable': var, **ci})
    if filas:
        display(pd.DataFrame(filas).round(3))

## 2. Kruskal-Wallis

In [ ]:
if 'tipo_entidad' in df.columns and df['tipo_entidad'].nunique() >= 2:
    res = []
    for var in VARS:
        if var not in df.columns:
            continue
        grupos = [g[var].dropna().values for _, g in df.groupby('tipo_entidad')]
        grupos = [g for g in grupos if len(g) >= 3]
        if len(grupos) >= 2:
            h, p = stats.kruskal(*grupos)
            res.append({'variable': var, 'H': round(h, 3), 'p': round(p, 4), 'significativo': p < 0.05})
    if res:
        display(pd.DataFrame(res))

## 3. Mann-Whitney

In [ ]:
if 'tipo_entidad' in df.columns and df['tipo_entidad'].nunique() >= 2:
    tipos = df['tipo_entidad'].unique().tolist()
    pares = []
    for var in VARS:
        if var not in df.columns:
            continue
        for i in range(len(tipos)):
            for j in range(i+1, len(tipos)):
                a = df[df['tipo_entidad'] == tipos[i]][var].dropna()
                b = df[df['tipo_entidad'] == tipos[j]][var].dropna()
                if len(a) >= 3 and len(b) >= 3:
                    u, p = stats.mannwhitneyu(a, b, alternative='two-sided')
                    pares.append({'variable': var, 'par': f'{tipos[i]} vs {tipos[j]}', 'p': round(p, 4), 'significativo': p < 0.05})
    if pares:
        display(pd.DataFrame(pares).sort_values('p').head(15))